In [ ]:
/Users/tatiana/Documents_new/pang/visor_freqk/summaries/results.ipynb

In [4]:
import os
import glob
from pathlib import Path
import pandas as pd

# Base results directory
# New layout: results/{sv_type}/{cov_err}/{SV_TYPE}/{size}/{freq}/{k}/
# cov_err encodes coverage and full error rate after the decimal (e.g. cov20_err0, cov20_err001)
results_dir = Path('/Users/tatiana/Documents_new/pang/visor_freqk/results')
results_dir = Path('../results')

# Find all allele frequency files recursively (any k value)
allele_freq_files = sorted(results_dir.rglob('*.allele_frequencies.k*.tsv'))

print(f"Found {len(allele_freq_files)} allele frequency files:\n")
print(f"{'SV':4s}  {'cov/err':12s}  {'size':6s}  {'freq':4s}  {'k':4s}  file")
print("-" * 75)

for f in allele_freq_files:
    parts = f.relative_to(results_dir).parts
    # parts: (sv_type, cov_err, SV_TYPE, size, freq, k, filename)
    sv_type = parts[2]   # 'DEL' or 'INS'
    cov_err = parts[1]   # 'cov50_err0'
    size    = parts[3]   # '100bp', '1kb', ...
    freq    = parts[4]   # 'f50'
    k       = parts[5]   # 'k31'
    print(f"{sv_type:4s}  {cov_err:12s}  {size:6s}  {freq:4s}  {k:4s}  {f.name}")

print(f"\nTotal: {len(allele_freq_files)} files")
print("\n" + "="*80)
print("Full paths:")
print("="*80)
for f in allele_freq_files:
    print(f)


Found 30 allele frequency files:

SV    cov/err       size    freq  k     file
---------------------------------------------------------------------------
DEL   cov20_err001  100bp   f50   k31   freq_100bp_f50_err001.allele_frequencies.k31.tsv
DEL   cov20_err001  10kb    f50   k31   freq_10kb_f50_err001.allele_frequencies.k31.tsv
DEL   cov20_err001  1kb     f50   k31   freq_1kb_f50_err001.allele_frequencies.k31.tsv
DEL   cov20_err001  500bp   f50   k31   freq_500bp_f50_err001.allele_frequencies.k31.tsv
DEL   cov20_err001  5kb     f50   k31   freq_5kb_f50_err001.allele_frequencies.k31.tsv
DEL   cov50_err0    100bp   f50   k31   freq_100bp_f50_err0.allele_frequencies.k31.tsv
DEL   cov50_err0    10kb    f50   k31   freq_10kb_f50_err0.allele_frequencies.k31.tsv
DEL   cov50_err0    1kb     f50   k31   freq_1kb_f50_err0.allele_frequencies.k31.tsv
DEL   cov50_err0    500bp   f50   k31   freq_500bp_f50_err0.allele_frequencies.k31.tsv
DEL   cov50_err0    5kb     f50   k31   freq_5kb_f50_err0.al

In [5]:
rows = []

for f in allele_freq_files:
    parts = f.relative_to(results_dir).parts
    # parts: (sv_root, cov_err, SV_TYPE, size, freq, k, filename)
    sv_root = parts[0]      # 'del' or 'ins'
    cov_err = parts[1]      # e.g. 'cov20_err0', 'cov20_err001'
    sv_type = parts[2]      # 'DEL' or 'INS'
    size    = parts[3]      # '100bp', '1kb', ...
    freq_lbl= parts[4]      # 'f50'
    k_lbl   = parts[5]      # 'k31'

    # Parse coverage / error from cov_err
    cov_part, err_part = cov_err.split('_')  # 'cov20', 'err0'
    coverage = int(cov_part.replace('cov', ''))
    error_rate = float(err_part.replace('err', '')) / 100.0  # 0 -> 0.0, 0.1 -> 0.001, etc.

    # Parse nominal allele frequency from label (e.g. f50 -> 0.5)
    freq_nominal = int(freq_lbl.replace('f', '')) / 100.0

    # Parse k from k-label (e.g. k31 -> 31)
    k = int(k_lbl.replace('k', ''))

    # Read AF line (it is a single line like '0.6|0.4')
    line = f.read_text().strip()
    parts_af = line.split('|')
    # Pad to at least 2 entries
    if len(parts_af) == 1:
        parts_af = parts_af + ['nan']
    af_ref = float(parts_af[0])
    af_alt = float(parts_af[1])

    rows.append({
        'sv_root': sv_root,
        'sv_type': sv_type,
        'cov_err': cov_err,
        'coverage': coverage,
        'error_rate': error_rate,
        'size': size,
        'freq_label': freq_lbl,
        'freq_nominal': freq_nominal,
        'k_label': k_lbl,
        'k': k,
        'file': str(f),
        'af_ref': af_ref,
        'af_alt': af_alt,
    })

results_df = pd.DataFrame(rows)
results_df

,sv_root,sv_type,cov_err,coverage,error_rate,size,freq_label,freq_nominal,k_label,k,file,af_ref,af_alt
0,del,DEL,cov20_err001,20,0.01,100bp,f50,0.5,k31,31,../results/del/cov20_err001/DEL/100bp/f50/k31/...,0.349414,0.650586
1,del,DEL,cov20_err001,20,0.01,10kb,f50,0.5,k31,31,../results/del/cov20_err001/DEL/10kb/f50/k31/f...,0.556867,0.443133
2,del,DEL,cov20_err001,20,0.01,1kb,f50,0.5,k31,31,../results/del/cov20_err001/DEL/1kb/f50/k31/fr...,0.456449,0.543551
3,del,DEL,cov20_err001,20,0.01,500bp,f50,0.5,k31,31,../results/del/cov20_err001/DEL/500bp/f50/k31/...,0.488409,0.511591
4,del,DEL,cov20_err001,20,0.01,5kb,f50,0.5,k31,31,../results/del/cov20_err001/DEL/5kb/f50/k31/fr...,0.386087,0.613913
5,del,DEL,cov50_err0,50,0.00,100bp,f50,0.5,k31,31,../results/del/cov50_err0/DEL/100bp/f50/k31/fr...,0.434015,0.565985
6,del,DEL,cov50_err0,50,0.00,10kb,f50,0.5,k31,31,../results/del/cov50_err0/DEL/10kb/f50/k31/fre...,0.429895,0.570105
7,del,DEL,cov50_err0,50,0.00,1kb,f50,0.5,k31,31,../results/del/cov50_err0/DEL/1kb/f50/k31/freq...,0.515843,0.484157
8,del,DEL,cov50_err0,50,0.00,500bp,f50,0.5,k31,31,../results/del/cov50_err0/DEL/500bp/f50/k31/fr...,0.565777,0.434223
9,del,DEL,cov50_err0,50,0.00,5kb,f50,0.5,k31,31,../results/del/cov50_err0/DEL/5kb/f50/k31/freq...,0.465494,0.534506


In [7]:
results_df['coverage'].value_counts()

coverage
50    20
20    10
Name: count, dtype: int64